In [1]:
import sys
import numpy as np
from time import time
from utils import get_default_train_test_sets, get_dataset_names_142, get_resample_indices
from sklearn.preprocessing import LabelEncoder
from pulsar import PULSAR
from sklearn.metrics import accuracy_score
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Function to run a toy example to trigger numba compilation
def run_toy_example_for_numba_compilation():
    print(f'Numba compiling starting....')
    start_time = time()
    classifier = PULSAR()
    X_toy = np.random.rand(10, 20)
    y_toy = np.random.randint(0, 2, 10)
    classifier.fit_transform(X_toy, y_toy)
    end_time = time()
    numba_compiling_time = end_time - start_time
    print(f'Numba compiling took: {numba_compiling_time} seconds') 


# Main function to run experiments
def get_classification_results_142datasets_30resamples(datasets_path, datasets_resample_indices_path, dataset_name):
    
    print(f'\n\nDataset {dataset_name}\n')

    number_of_resamples = 30  # 30 resamples per dataset


    # Load dataset
    X_train, y_train, X_test, y_test = get_default_train_test_sets(datasets_path, dataset_name)
    X = np.concatenate((X_train, X_test), axis=0)
    y = np.concatenate((y_train, y_test), axis=0)

    resample_accuracies = []
    train_times = []
    test_times = []

    for resample_id in range(0, number_of_resamples):
        print(f'\n\nResample {resample_id} of {dataset_name}')

        # Load resampled dataset
        train_indices = get_resample_indices(datasets_resample_indices_path, dataset_name, "TRAIN", resample_id)
        test_indices = get_resample_indices(datasets_resample_indices_path, dataset_name, "TEST", resample_id)
        X_train_resampled = X[train_indices]
        X_test_resampled = X[test_indices]
        y_train_resampled = y[train_indices]
        y_test_resampled = y[test_indices]


        n_classes = len(np.unique(y_train_resampled))

        # Check if X in 3D make it 2D
        if X_train_resampled.ndim == 3:
            X_train_resampled = X_train_resampled.reshape(X_train_resampled.shape[0], -1)
            X_test_resampled = X_test_resampled.reshape(X_test_resampled.shape[0], -1)

        # Encode labels
        le = LabelEncoder()
        y_train_resampled = le.fit_transform(y_train_resampled)
        y_test_resampled = le.transform(y_test_resampled)


        # Initialize classifier with default parameters
        classifier = PULSAR()

        # Alternatively, you can customize the parameters as needed. Below is an example using the default settings:
        # classifier = PULSAR(
        #     bake_off_classifiers=['ridge', 'extra_trees'],
        #     time_series_representations=['original', 'periodogram', 'derivative', 'autoregressive'],
        #     local_statistics=['mean', 'stdev', 'slope', 'min', 'max', 'iqr', 'median'],
        #     global_statistics=['max_pooling', 'mean_pooling', 'min_pooling', 'median_pooling', 
        #                         'iqr_pooling', 'stdev_pooling', 'mean_crossing_pooling', 
        #                         'values_above_mean_pooling', 'slope_pooling',
        #                         ],
        #     list_interval_lengths=[7, 9, 11],
        #     depth_local_features=4,
        #     percentage_top_local_features=40,
        #     num_random_selected_pooling_operators_per_interval=6,
        #     max_dilation=16,
        # )

        # Train classifier
        start_time = time()
        classifier.fit(X_train_resampled, y_train_resampled)
        train_time = time() - start_time
        print(f'Train time (seconds): {train_time}')
        train_times.append(train_time)

        # Predict
        start_time = time()
        y_pred = classifier.predict(X_test_resampled)
        test_time = time() - start_time
        print(f'Test time (seconds): {test_time}')
        test_times.append(test_time)

        # Compute accuracy
        accuracy = accuracy_score(y_test_resampled, y_pred)
        print(f'Accuracy: {accuracy}')
        resample_accuracies.append(accuracy)


    # Save results to CSV
    results_dict = {
        'dataset': [dataset_name],
        'train size': [X_train.shape[0]],
        'test size': [X_test.shape[0]],
        'length': [X_train.shape[-1]],
        'nclasses': [len(le.classes_)],
    }
    # Add resample accuracy columns
    for i, acc in enumerate(resample_accuracies):
        results_dict[f'resample_{i}'] = acc

    # Add average metrics
    results_dict['average_accuracy'] = np.mean(resample_accuracies)
    results_dict['average_train_time'] = np.mean(train_times)
    results_dict['average_test_time'] = np.mean(test_times)

    results = pd.DataFrame(results_dict)

    results_file = f'temp_results/PULSAR__{number_of_resamples}resamples.csv'
    file_exists = os.path.isfile(results_file)

    results.to_csv(results_file, mode='a', index=False, header=not file_exists)  # Append results without rewriting the header


In [ ]:
datasets_path = "142_Univariate_Time_Series_Datasets"
datasets_resample_indices_path = "ResampleIndices"

# Get dataset name
# dataset_names = get_dataset_names_142(datasets_path) # Using all 142 datasets. Needs to the corresponding resample indices files.
dataset_names = ["Adiac","Car","ItalyPowerDemand"] #Using the datasets provided in the 142_Univariate_Time_Series_Datasets folder for testing


## Run a transformation on a toy dataset to compile numba
run_toy_example_for_numba_compilation()

for dataset_name in dataset_names:
    get_classification_results_142datasets_30resamples(datasets_path, datasets_resample_indices_path, dataset_name)
    print(f'\nFinished dataset: {dataset_name}\n\n')
print("All datasets processed.")